# Cosine few-shot evaluation

Loads a checkpoint produced by a pretrain experiment and runs prototypical-
network few-shot evaluation. Everything for this run is written to
`experiments/<experiment_name>/evaluations/<eval_name>/`:

- `eval_config.json` — every parameter you set below + the architecture
  auto-loaded from the pretrain experiment (under `arch_from_pretrain`)
- `eval_metadata.json` — timestamp, git SHA, OA/AA/Kappa summary
- `results.json` — full results including per-class accuracy
- `eval.log` — evaluation log
- `plots/` — confusion matrix, per-class accuracy, t-SNE, ...

The `experiment_name` is the key: architecture (`embed_dim`, `num_heads`,
`num_layers`, `patch_size`, `lambda_factor`, `dropout`, projection-head
settings) is read automatically from `experiments/<name>/pretrain_config.yaml`,
so `eval_params` only needs evaluation-shaped knobs.

Multiple evals against the same experiment are normal — just pick a different
`eval_name` each time.

In [ ]:
# === Parameters ===
# Which pretraining experiment to evaluate.
experiment_name = "houston_enhanced_v1"

# Short name for THIS evaluation run — becomes the eval subdir.
eval_name = "houston_5way_5shot_cosine_t10"

# Checkpoint epoch to load. None = latest in checkpoints/ (or checkpoint_final.pth if present).
epoch = None

# Alternative: pass an explicit checkpoint path (or the string 'random' for an untrained baseline).
# Overrides `epoch` when set.
checkpoint = None

# Evaluation hyperparameters. Architecture (embed_dim, num_heads, num_layers,
# patch_size, lambda_factor, dropout, proj_*) is auto-loaded from the pretrain
# experiment — only set those keys here if you deliberately want to deviate.
eval_params = {
    "dataset": "houston",                  # "houston" | "trento" | "muufl"
    "n_way": 5,
    "k_shot": 5,
    "k_query": 15,
    "num_episodes": 600,
    "distance_metric": "cosine",          # "cosine" | "euclidean"
    "temperature": 10.0,
    "prototype_mode": "mean_features",    # "mean_features" | "mean_distances"
    "pool_sigma": None,                    # e.g. 2.0 for center-weighted pooling
    "split": "test",
    "seed": 42,
    "no_plots": False,
    "num_example_episodes": 3,
    "max_tsne_samples": 0,

    # --- Architecture overrides (rare; defaults come from the pretrain experiment) ---
    # "embed_dim": 128, "num_heads": 2, "num_layers": 2,
    # "patch_size": 11, "lambda_factor": 0.5, "dropout": 0.1,
    # "use_projection": True,
}

overwrite = False

In [ ]:
import os, sys
from pathlib import Path
REPO = Path.cwd()
while not (REPO / "lib" / "experiments.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("Could not locate repo root containing lib/experiments.py")
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Repo root:", REPO)

In [ ]:
from lib.eval_runner import run_evaluation

ev = run_evaluation(
    experiment_name=experiment_name,
    eval_name=eval_name,
    epoch=epoch,
    checkpoint=checkpoint,
    eval_params=eval_params,
    overwrite=overwrite,
)
print("Eval dir:", ev.root)

In [ ]:
# Summary metrics from this run.
import json
print(json.dumps(ev.metadata.get("summary", {}), indent=2, default=str))
print("\nArchitecture loaded from pretrain experiment:")
print(json.dumps(ev.config.get("arch_from_pretrain", {}), indent=2))

In [ ]:
# Quick look at per-class accuracy from the full results.json.
import json
results = json.loads(ev.results_path.read_text())
per_class = results.get("per_class", {})
for cls, data in sorted(per_class.items(), key=lambda kv: int(kv[0])):
    print(f"class {cls}: {data['accuracy']:6.2f}%  \u00b1 {data['ci_95']:5.2f}%  (n={data['total_samples']})")